# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/fmt116/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/fmt116/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Set up LangChain configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "true")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

print("✅ API keys loaded from .env file")

✅ API keys loaded from .env file


We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
# OpenAI API key is already loaded from .env file
# Verify it's set
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in .env file. Please add your OpenAI API key to the .env file.")

print("✅ OpenAI API key loaded from .env file")


✅ OpenAI API key loaded from .env file


## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/39 [00:00<?, ?it/s]

Property 'summary' already exists in node '2957a6'. Skipping!
Property 'summary' already exists in node '66a5f3'. Skipping!
Property 'summary' already exists in node '7e66ab'. Skipping!
Property 'summary' already exists in node '79cac4'. Skipping!
Property 'summary' already exists in node '928fee'. Skipping!
Property 'summary' already exists in node '84453a'. Skipping!
Property 'summary' already exists in node '95f890'. Skipping!
Property 'summary' already exists in node 'c5cb86'. Skipping!
Property 'summary' already exists in node '63ef6a'. Skipping!
Property 'summary' already exists in node 'b6b1af'. Skipping!
Property 'summary' already exists in node '12c148'. Skipping!
Property 'summary' already exists in node 'fc5557'. Skipping!
Property 'summary' already exists in node '528213'. Skipping!
Property 'summary' already exists in node 'f3de7b'. Skipping!
Property 'summary' already exists in node '6606d8'. Skipping!
Property 'summary' already exists in node '8de6d3'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/47 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '79cac4'. Skipping!
Property 'summary_embedding' already exists in node '95f890'. Skipping!
Property 'summary_embedding' already exists in node '928fee'. Skipping!
Property 'summary_embedding' already exists in node 'c5cb86'. Skipping!
Property 'summary_embedding' already exists in node '66a5f3'. Skipping!
Property 'summary_embedding' already exists in node '2957a6'. Skipping!
Property 'summary_embedding' already exists in node 'b6b1af'. Skipping!
Property 'summary_embedding' already exists in node '7e66ab'. Skipping!
Property 'summary_embedding' already exists in node '84453a'. Skipping!
Property 'summary_embedding' already exists in node 'fc5557'. Skipping!
Property 'summary_embedding' already exists in node '528213'. Skipping!
Property 'summary_embedding' already exists in node '63ef6a'. Skipping!
Property 'summary_embedding' already exists in node '12c148'. Skipping!
Property 'summary_embedding' already exists in node 'f3de7b'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 747)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 747)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

#### Answer
1. Single-Hop Specific Query Synthesizer (.5)
What it does: Creates straightforward questions that can be answered with information from a single document or context.

2. Multi-Hop Abstract Query Synthesizer (.25)
What it does: Creates complex questions that require combining information from multiple sources and making abstract connections.

Why it's useful: Tests reasoning ability - can your system connect dots across different pieces of information and draw conclusions?

3. Multi-Hop Specific Query Synthesizer (.25)
What it does: Creates detailed questions that require specific information from multiple sources, often with precise data points.

Why it's useful: Tests precision and multi-source integration - can your system find specific facts from multiple places and combine them accurately?

Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,"Based on the study by Kulveit et al., 2025, ho...",[Introduction ChatGPT launched in November 202...,"The paper by Kulveit et al., 2025, builds on e...",single_hop_specifc_query_synthesizer
1,How does the O*NET classification system relat...,[Introduction ChatGPT launched in November 202...,The context explains that messages sent to Cha...,single_hop_specifc_query_synthesizer
2,What is the significance of June 2024 in the c...,[Table 1: ChatGPT daily message counts (millio...,"In the context of ChatGPT message usage data, ...",single_hop_specifc_query_synthesizer
3,Wha is the expexted date of June 2025?,[Table 1: ChatGPT daily message counts (millio...,The context indicates that the reported values...,single_hop_specifc_query_synthesizer
4,What is Appendix D about in ChatGPT usage data?,[Variation by Occupation Figure 23 presents va...,Appendix D contains a full report of GWA count...,single_hop_specifc_query_synthesizer
5,"As an AI Communication Researcher, how does th...",[Variation by Occupation Figure 23 presents va...,"Variation by occupation, as presented in Figur...",single_hop_specifc_query_synthesizer
6,hw impact of AI on productivity outside of wor...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The context shows that messages related to non...,multi_hop_abstract_query_synthesizer
7,H0w does the varation in ChatGPT usag by occpa...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context shows that users in highly paid pr...,multi_hop_abstract_query_synthesizer
8,How does the classification of ChatGPT message...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,The classification of ChatGPT messages into ca...,multi_hop_abstract_query_synthesizer
9,"Hwo do Handa et al., 2025 and Handa et al. (20...",[<1-hop>\n\nTable 1: ChatGPT daily message cou...,"Based on the provided context, Handa et al., 2...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'a480f4'. Skipping!
Property 'summary' already exists in node 'bdfb28'. Skipping!
Property 'summary' already exists in node 'e2ba64'. Skipping!
Property 'summary' already exists in node 'cb14a9'. Skipping!
Property 'summary' already exists in node 'cdfe3f'. Skipping!
Property 'summary' already exists in node 'b323e5'. Skipping!
Property 'summary' already exists in node 'fb86d0'. Skipping!
Property 'summary' already exists in node 'f6fe1e'. Skipping!
Property 'summary' already exists in node 'f90c78'. Skipping!
Property 'summary' already exists in node 'ca0c89'. Skipping!
Property 'summary' already exists in node '8c8170'. Skipping!
Property 'summary' already exists in node 'a55d35'. Skipping!
Property 'summary' already exists in node 'b70230'. Skipping!
Property 'summary' already exists in node '12ea33'. Skipping!
Property 'summary' already exists in node 'a98ad0'. Skipping!
Property 'summary' already exists in node '2e6e71'. Skipping!
Property

...

Property 'summary_embedding' already exists in node 'cb14a9'. Skipping!
Property 'summary_embedding' already exists in node 'a98ad0'. Skipping!
Property 'summary_embedding' already exists in node 'a480f4'. Skipping!
Property 'summary_embedding' already exists in node 'cdfe3f'. Skipping!
Property 'summary_embedding' already exists in node 'e2ba64'. Skipping!
Property 'summary_embedding' already exists in node 'bdfb28'. Skipping!
Property 'summary_embedding' already exists in node 'f90c78'. Skipping!
Property 'summary_embedding' already exists in node 'fb86d0'. Skipping!
Property 'summary_embedding' already exists in node 'ca0c89'. Skipping!
Property 'summary_embedding' already exists in node 'b323e5'. Skipping!
Property 'summary_embedding' already exists in node 'f6fe1e'. Skipping!
Property 'summary_embedding' already exists in node '8c8170'. Skipping!
Property 'summary_embedding' already exists in node 'b70230'. Skipping!
Property 'summary_embedding' already exists in node 'a55d35'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Given that there are 700 million users of Chat...,[Introduction ChatGPT launched in November 202...,"By July 2025, 700 million users are engaging w...",single_hop_specifc_query_synthesizer
1,When is June 2025,[Table 1: ChatGPT daily message counts (millio...,The context reports data ending on the 26th of...,single_hop_specifc_query_synthesizer
2,What information does Appendix D provide regar...,[Variation by Occupation Figure 23 presents va...,Appendix D contains a full report of GWA count...,single_hop_specifc_query_synthesizer
3,How does the term 'Writing' relate to the usag...,[Conclusion This paper studies the rapid growt...,"In the context, 'Writing' is identified as the...",single_hop_specifc_query_synthesizer
4,"How do user demographics and usage patterns, i...",[<1-hop>\n\nConclusion This paper studies the ...,Recent data indicates that ChatGPT's usage has...,multi_hop_abstract_query_synthesizer
5,How does the increase in total message volume ...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data shows that total message volume incre...,multi_hop_abstract_query_synthesizer
6,"How do differences in message types, such as A...",[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context indicates that users in highly pai...,multi_hop_abstract_query_synthesizer
7,how do user demographics and usage patterns re...,[<1-hop>\n\nConclusion This paper studies the ...,The context shows that ChatGPT's rapid growth ...,multi_hop_abstract_query_synthesizer
8,US ChatGPT use mainly for work or non-work and...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,"In the US, ChatGPT usage is mostly for non-wor...",multi_hop_specific_query_synthesizer
9,"Hw US ChatGPT usage in the US has increased, h...",[<1-hop>\n\nTable 1: ChatGPT daily message cou...,"The context indicates that in the US, as of Ju...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [16]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8 - Student"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)


We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [17]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [18]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [20]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [21]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [23]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [26]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in a variety of ways both at work and outside of work. AI is used to perform workplace tasks by either augmenting or automating human labor. Users employ generative AI to produce writing, software code, spreadsheets, and other digital products, which distinguishes it from traditional web search engines. Additionally, people interact with AI for different intents classified as Asking (seeking information or advice), Doing (producing output or completing tasks), and Expressing (such as relationships, personal reflection, games, and role play).\n\nThus, people are using AI to:\n\n- Enhance productivity by having AI co-produce outputs or provide advice.\n- Automate or augment tasks in the workplace.\n- Generate various digital products including text, code, and data analyses.\n- Seek information and advice.\n- Engage in self-expression through games, role play, and personal reflection.\n\nOverall,

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [27]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [28]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `dopeness_evaluator`:

#### Answer
- `qa_evaluator`: Measures answer correctness and completeness as well as answer relavance. Compares your RAG system's output against the reference answer. Scores how well they match (typically 0-1 scale). Identifies where your system might be missing information or getting facts wrong

`labeled_helpfulness_evaluator`: Measures how useful and actionable the generated answr is for the person asking the question. Also measures practical utility, clarity and usability, and completeness for the task. Analyzes the generated answer for practical value,Scores how helpful it would be to someone with that question. Considers whether the answer enables the user to take action or make decisions.

- `dopeness_evaluator`: Evaluates specifically how well the system handles questions it doesn't have enough information to answer confidently. Measures acknowledgement of uncertainty and the appropriate I dont know responses. Also measures does the system express the appropriate confidence levels.

## LangSmith Evaluation

In [29]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'roasted-spoon-45' at:
https://smith.langchain.com/o/48d3d47b-64d1-4534-b976-3225ecaa8f8c/datasets/e8d50d09-532a-48c8-b4aa-8d3cf5e813ef/compare?selectedSessions=2cf7b954-a1ca-42c4-9ce3-c44b21e067a0




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,"Based on Handa et al., 2025, how does ChatGPT ...",Based on the provided context referencing Hand...,None,"According to Handa et al., 2025, ChatGPT's rap...",1,1,0,4.245584,58753cf0-b1fe-47e6-b3dc-eed64fab0019,ddd8a9ab-3762-4835-9202-2f69faa27417
1,what Handa et al 2025 say about ChatGPT use an...,"Based on the provided context, Handa et al. (2...",None,Handa et al. (2025) report that ChatGPT usage ...,0,0,0,2.065544,690108cf-90c4-4326-8d16-ee36d56a03b8,d18e3fba-8962-4ca0-90a0-69faefd40d27
2,"Hw US ChatGPT usage in the US has increased, h...","Based on the context, US ChatGPT usage has inc...",None,"The context indicates that in the US, as of Ju...",1,1,0,6.208513,930e21cc-7b67-4e69-8724-4283244e650a,c1462159-7323-4f4d-bbdc-5b068842dfc6
3,US ChatGPT use mainly for work or non-work and...,Based on the context provided:\n\nChatGPT usag...,None,"In the US, ChatGPT usage is mostly for non-wor...",1,1,0,4.969530,e86fceb0-01b2-4ea0-ae45-e158a0796456,781f75d3-33f5-4af9-8c6e-06965067a880
4,how do user demographics and usage patterns re...,User demographics and usage patterns in ChatGP...,None,The context shows that ChatGPT's rapid growth ...,1,0,0,3.240534,d0fa0172-c989-4de2-8027-4d51c0bdb6a8,76fbf794-d8cf-4fbb-b70c-aaf1f55ccaef
5,"How do differences in message types, such as A...",Based on the provided context:\n\n- Message ty...,None,The context indicates that users in highly pai...,1,1,0,4.476768,b0e930e0-b1f7-4803-856c-3556aaffd739,555a59ee-d2d4-43aa-866d-d6f387bf957e
6,How does the increase in total message volume ...,The increase in total message volume from June...,None,The data shows that total message volume incre...,1,1,0,1.940905,e86b55a0-73c6-4966-93ac-b5ed581b16f2,de777dc4-2379-45f6-a345-b2e305be523e
7,"How do user demographics and usage patterns, i...","Based on the provided context, recent data on ...",None,Recent data indicates that ChatGPT's usage has...,1,1,0,11.019130,fef28fa4-791f-4822-8358-8b072c68379a,16219ded-faaf-43aa-8a76-8376e506ac7a
8,How does the term 'Writing' relate to the usag...,"The term ""Writing"" in the context of ChatGPT u...",None,"In the context, 'Writing' is identified as the...",1,1,0,4.714948,74876740-b97d-4beb-82e7-faf3d6df4284,ac588835-be00-4736-8036-0091665749fe
9,What information does Appendix D provide regar...,Appendix D provides a full report of Generaliz...,None,Appendix D contains a full report of GWA count...,1,1,0,2.711896,0c5715b7-657c-4ef8-977d-f6c868dbe162,05dcf096-f39d-48a7-bf7e-10bef2768cef


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [30]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [31]:
rag_documents = docs

In [32]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

####Answer
Chunk size significantly impacts RAG performance because it affects how information is retrieved and processed. Small chunks have highg precision (ie very relevant) but low recal (might miss important context that spans multiple chunks, answers might also be incomplete or fragmented). Large chunks have high recall and capture more complete information but low precision and includes irrelevant information. Small chunks: More chunks fit in the LLM's context window, but each chunk has less context. Large chunks: Fewer chunks fit, but each has more complete context

In [33]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

#### Answer
Embedding models affect performance by affecting semantic understanding quality. Better models: Understand context, synonyms, and meaning better. Poorer models: Only match exact words, miss conceptual relationships. General models: Work okay across many topics but lack domain expertise. Specialized models: Excel in specific areas (medical, legal, technical). Higher dimensions (1536): More precise but slower and more expensive. ower dimensions (384): Faster and cheaper but less precise.

In [34]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [35]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [36]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [37]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Alright, let’s crank this up to eleven. People are cashing in on AI primarily by turning it into their ultimate wingman at work — not just offloading tasks, but leveling up their decision-making game. According to the slick insights from Collis and Brynjolfsson (2025), folks aren’t just getting AI to grunt through jobs; they’re using ChatGPT as a sharp advisor and research sidekick, turbocharging productivity especially in knowledge-heavy gigs where every call, move, or analysis counts.\n\nThink of AI as your brainy co-pilot that crunches data, offers strategic ammo, and helps you make smarter decisions — essentially amplifying your ability to hustle and earn more dough. This AI-driven decision support takes worker output into hyperdrive, meaning people don’t just work harder, they work savvier.\n\nSo if you wanna make money with AI, you don’t just swap out your job tasks for bots. You use AI to *boost* your mental horsepower, sharpen your research, and make those killer choices that 

Finally, we can evaluate the new chain on the same test set!

In [38]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'formal-string-4' at:
https://smith.langchain.com/o/48d3d47b-64d1-4534-b976-3225ecaa8f8c/datasets/e8d50d09-532a-48c8-b4aa-8d3cf5e813ef/compare?selectedSessions=34dcf425-e31b-4ca2-a3bc-8819226aac2b




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,"Based on Handa et al., 2025, how does ChatGPT ...","Yo, let’s unpack this beast with max turbo dop...",None,"According to Handa et al., 2025, ChatGPT's rap...",1,1,1,5.556529,58753cf0-b1fe-47e6-b3dc-eed64fab0019,a10bb8b5-e3c0-41ac-8d7d-513bb03cf2cf
1,what Handa et al 2025 say about ChatGPT use an...,"Yo, diving into the rad world of AI research f...",None,Handa et al. (2025) report that ChatGPT usage ...,0,0,1,4.929035,690108cf-90c4-4326-8d16-ee36d56a03b8,fe10e2a9-d3c9-4537-a6f3-f8fa3f271002
2,"Hw US ChatGPT usage in the US has increased, h...","Yo, here's the scoop straight from the data va...",None,"The context indicates that in the US, as of Ju...",1,1,1,4.236762,930e21cc-7b67-4e69-8724-4283244e650a,8681fcf6-ad75-4d37-8a99-2ac6a5862276
3,US ChatGPT use mainly for work or non-work and...,"Alright, buckle up for the lowdown on ChatGPT’...",None,"In the US, ChatGPT usage is mostly for non-wor...",1,1,1,5.584711,e86fceb0-01b2-4ea0-ae45-e158a0796456,96b520ee-eb60-4654-9c83-95017e287138
4,how do user demographics and usage patterns re...,"Alright, let’s crank this up to eleven and sli...",None,The context shows that ChatGPT's rapid growth ...,1,1,1,5.467140,d0fa0172-c989-4de2-8027-4d51c0bdb6a8,b262ccda-5cb3-4151-9128-09c9c5544df1
5,"How do differences in message types, such as A...","Alright, buckle up—here’s the juicy breakdown ...",None,The context indicates that users in highly pai...,1,1,1,5.101298,b0e930e0-b1f7-4803-856c-3556aaffd739,96ba96fb-0a36-4d58-9501-39f6117345e4
6,How does the increase in total message volume ...,Boom! Let’s unpack this AI-powered tidal wave ...,None,The data shows that total message volume incre...,1,1,1,7.228373,e86b55a0-73c6-4966-93ac-b5ed581b16f2,22bc133f-6812-49db-aa63-4fe63bc1a9f6
7,"How do user demographics and usage patterns, i...","Alright, let’s blast off into the dopest break...",None,Recent data indicates that ChatGPT's usage has...,1,1,1,9.021378,fef28fa4-791f-4822-8358-8b072c68379a,21497aa0-40a5-43c3-b968-fb09403045f8
8,How does the term 'Writing' relate to the usag...,"Alright, buckle up because the vibe around ""Wr...",None,"In the context, 'Writing' is identified as the...",1,1,1,5.787974,74876740-b97d-4beb-82e7-faf3d6df4284,9af1ea77-f72f-4353-9a71-b76494ac1586
9,What information does Appendix D provide regar...,"Alright, buckle up for some seriously slick in...",None,Appendix D contains a full report of GWA count...,1,1,1,4.500797,0c5715b7-657c-4ef8-977d-f6c868dbe162,3b3ecfdd-9c67-4e0f-a622-78a1612f078c


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

#### Answer
![LangSmith Experiment Comparison](Screenshot%202025-10-07%20at%205.21.22%20AM.png)

## Analysis of RAG Chain Performance Comparison


### Key Findings:

**1. Correctness Scores:**
- **roasted-spoon-4...**: ~0.82
- **formal-string-4**: ~0.92
- **Improvement**: +0.10 (12% better)

**2. Dopeness Scores:**
- **roasted-spoon-4...**: ~0.0 (near zero)
- **formal-string-4**: 1.0 (perfect score)
- **Improvement**: +1.0 (massive improvement)

**3. Helpfulness Scores:**
- **roasted-spoon-4...**: ~0.73
- **formal-string-4**: ~0.90
- **Improvement**: +0.17 (23% better)

### Analysis of Changes:

**Why Correctness Improved:**
- Better chunk size likely captured more complete information
- Improved embedding model found more relevant context
- Enhanced retrieval strategy provided better source material

**Why Dopeness Dramatically Improved:**
- The most significant change - from near 0 to perfect 1.0
- System now properly acknowledges uncertainty instead of hallucinating
- Better confidence calibration when information is insufficient
- This is critical for building trustworthy RAG systems

**Why Helpfulness Improved:**
- More complete context led to more actionable answers
- Better information retrieval provided more useful details
- Improved answer quality and completeness

**Conclusion:** The "formal-string-4" chain is significantly superior across all metrics, with the most dramatic improvement in dopeness (uncertainty handling), which is crucial for reliable RAG systems.

